In this part, we want to crawl recipes from the 'allrecipes' website and extract the spices for each recipe. To achieve this, we divided the process into two steps:
1.   First, we crawl the recipes along with their ingredients and save them in a CSV file (recipes_ingredients.csv).
2.   Then, we filter the ingredients to keep only the spices, and save these in a new CSV file (recipes_spices.csv).

In [ ]:
# Connection to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Imports
import os, re, json, time, gzip, random, csv, itertools, warnings
import xml.etree.ElementTree as ET
import requests
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter
from bs4 import BeautifulSoup
from tqdm import tqdm

import pandas as pd

1. **Crawl the recipes and its ingredients**

First, we set the configuration parameters (number of recipes, timeout, sleep interval, output path), and initialized an HTTP session with a user-agent and retry logic for stable crawling.

In [ ]:
# ------------------- CONFIG -------------------
TARGET_RECIPES = 5000     # how many recipes to collect
SITEMAP_MULT   = 2      # multiply by this to fetch more URLs (to offset failures)
TIMEOUT        = 20     # request timeout in seconds
JITTER_RANGE   = (0.4, 1.0)  # random sleep between requests

OUT_INGS = "/content/drive/MyDrive/Data Mining - Final Project/recipes_ingredients.csv"


# ------------------- HTTP SESSION -------------------
session = requests.Session()
session.headers["User-Agent"] = "Mozilla/5.0 (compatible; ingredient-scraper/0.1)"

# Retry logic for failed requests
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=3, backoff_factor=0.6, status_forcelist=[429,500,502,503,504]
)))

def sleep_a_bit():
    """Pause between requests to avoid hammering the server."""
    time.sleep(random.uniform(*JITTER_RANGE))

Now, we define helper functions for extracting recipe IDs, loading recipe URLs from sitemaps, parsing structured data, and normalizing cuisines.

In [ ]:
# ------------------- HELPERS -------------------
def recipe_id(u: str) -> str:
    """Extract numeric recipe ID from a recipe URL."""
    try:
        return u.rstrip("/").split("/recipe/")[1].split("/")[0]
    except Exception:
        return u.rstrip("/").split("/")[-1]

def load_recipe_urls():
    """Fetch recipe URLs from all AllRecipes sitemaps."""
    idx = "https://www.allrecipes.com/sitemap.xml"
    try:
        r = session.get(idx, timeout=TIMEOUT); r.raise_for_status()
        root = ET.fromstring(r.text)
    except Exception as e:
        warnings.warn(f"sitemap index fetch failed: {e}")
        return []

    ns = {'sm':'http://www.sitemaps.org/schemas/sitemap/0.9'}
    urls = []

    # Iterate through sub-sitemaps
    for sm in root.findall('sm:sitemap', ns):
        loc = sm.findtext('sm:loc', default="", namespaces=ns).strip()
        if not loc:
            continue
        try:
            data = session.get(loc, timeout=TIMEOUT).content
            xml  = gzip.decompress(data) if loc.endswith(".gz") else data
            doc  = ET.fromstring(xml)

            # Collect only recipe URLs
            for u in doc.findall('sm:url', ns):
                locu = u.findtext('sm:loc', default="", namespaces=ns).strip()
                if "/recipe/" in locu:
                    urls.append(locu)
        except Exception:
            continue
    return urls


def parse_jsonld(soup):
    """Extract Recipe JSON-LD blocks (structured data inside <script>)."""
    out = []
    for tag in soup.find_all("script", type="application/ld+json"):
        if not tag.string:
            continue
        try:
            data = json.loads(tag.string)
        except Exception:
            continue

        # Traverse nested dicts/lists
        stack = [data]
        while stack:
            cur = stack.pop()
            if isinstance(cur, dict):
                typ = cur.get("@type")
                if typ == "Recipe" or (isinstance(typ, list) and "Recipe" in typ):
                    out.append(cur)
                for v in cur.values():
                    if isinstance(v, (dict, list)):
                        stack.append(v)
            elif isinstance(cur, list):
                stack.extend(cur)
    return out

def normalize_cuisine(c):
    """Normalize cuisine label (lowercase, strip, unify)."""
    if not c:
        return None
    if isinstance(c, list):
        # drop generic labels like "world cuisine"
        c = [x for x in c if str(x).strip().lower() not in {"world cuisine","recipes"}]
        c = c[0] if c else None
    else:
        c = str(c).strip()

    if not c or c.lower() in {"world cuisine","recipes"}:
        return None

    # normalize
    c = c.lower().strip()
    c = re.sub(r"\bcuisine\b", "", c)  # remove the word 'cuisine'
    c = c.replace(" ", "_")            # replace spaces with underscores (to avoid possible problems)
    return c

def split_ingredient(ing: str):
    """
    Split ingredient text into separate items if it contains 'and'.
    Example:
        'salt and pepper, to taste' -> ['salt', 'pepper, to taste']
    """
    ing = ing.strip()
    if " and " in ing.lower():
        parts = ing.split(" and ")
        # Strip whitespace from each part
        return [p.strip() for p in parts if p.strip()]
    else:
        return [ing]

def extract_ingredients(html, url):
    """Parse HTML, return dict with name, url, cuisine, ingredients."""
    soup = BeautifulSoup(html, "lxml")
    blocks = parse_jsonld(soup)

    names, cuisines, ing_list = [], [], []

    # Extract from JSON-LD if available
    for r in blocks:
        ings = r.get("recipeIngredient") or []
        ings = [x.strip() for x in ings if isinstance(x, str) and x.strip()]
        if ings:
            ing_list.extend(ings)
        nm = r.get("name")
        if nm:
            names.append(nm.strip())
        cu = normalize_cuisine(r.get("recipeCuisine"))
        if cu:
            cuisines.append(cu)

    if ing_list:
        # Deduplicate ingredients (case-insensitive)
        seen = set(); dedup_ings = []
        for x in ing_list:
          for sub in split_ingredient(x):
              if sub.lower() not in seen:
                  seen.add(sub.lower())
                  dedup_ings.append(sub)

        # Choose best recipe name
        name = max(names, key=len) if names else (soup.title.string.strip() if soup.title else None)
        cuisine = cuisines[0] if cuisines else None
        return {"name": name, "url": url, "cuisine": cuisine, "ingredients": dedup_ings}

    # fallback HTML
    html_ings = [t.get_text(strip=True) for t in soup.select(
        ".ingredients-item-name, .mntl-structured-ingredients__list-item")]
    if html_ings:
        name_el = soup.select_one("h1")
        name = name_el.get_text(strip=True) if name_el else (soup.title.string.strip() if soup.title else None)
        return {"name": name, "url": url, "cuisine": None, "ingredients": html_ings}

    return None

Here the main function that coordinates the crawling process: it collects recipes, extracts their ingredients, and saves the results into a CSV file.

In [ ]:
# ------------------- MAIN -------------------
def build_ingredient_csv():
    """Crawl recipes and save to CSV: id | cuisine | recipe_name | recipe_url | ingredient"""

    # Create CSV
    with open(OUT_INGS, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(["id","cuisine","recipe_name","recipe_url","ingredient"])

    # Load URLs from sitemaps
    urls = load_recipe_urls()
    random.shuffle(urls)  # shuffle to avoid bias
    sample_n = TARGET_RECIPES * SITEMAP_MULT

    seen_recipe = set()
    got = 0
    recipe_id_counter = 1

    # Crawl recipes
    for url in tqdm(itertools.islice(urls, sample_n), total=sample_n, desc="Crawling"):
        if got >= TARGET_RECIPES:
            break
        try:
            r = session.get(url, timeout=TIMEOUT); r.raise_for_status()
        except Exception:
            sleep_a_bit(); continue

        info = extract_ingredients(r.text, url)
        sleep_a_bit()
        if not info or not info["ingredients"] or not info["cuisine"]:
            continue  # skip recipes with no ingredients or no cuisine

        # use URL-based id only to avoid duplicates
        rid = recipe_id(url)
        if rid in seen_recipe:
            continue

        # sanity check: no empty fields
        if not info["name"] or not info["url"]:
            continue

        # Append to CSV
        with open(OUT_INGS, "a", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            for ing in info["ingredients"]:
                if ing.strip():  # avoid empty ingredient
                    w.writerow([
                        recipe_id_counter,
                        info["cuisine"],
                        info["name"],
                        info["url"],
                        ing.strip()
                    ])

        seen_recipe.add(rid)
        got += 1
        recipe_id_counter += 1  # increment for next recipe

    print(f"\nDone. Recipes collected: {got}")
    print(f"Saved: {OUT_INGS}")

# ---------- RUN ----------
if __name__ == "__main__":
    build_ingredient_csv()

Crawling:  75%|███████▍  | 7477/10000 [2:13:53<45:10,  1.07s/it]


Done. Recipes collected: 5000
Saved: /content/drive/MyDrive/Data Mining - Final Project/recipes_ingredients.csv


2. **Recipes and spices**

First, we set the file paths and load the spice mapping JSON (spices_map.json) that we created.
Then, to maximize the accuracy of spice matching, we expand each spice and its synonyms into all common word forms, build a reverse lookup from synonym to canonical name, and compile a regex that can detect any spice variant in recipe text.

In [ ]:
# ---------- CONFIG ----------
# File paths
SPICE_MAP_PATH   = "/content/drive/MyDrive/Data Mining - Final Project/spices json/spices_map.json"
RECIPE_INGS_PATH = "/content/drive/MyDrive/Data Mining - Final Project/recipes_ingredients.csv"
OUT_SPICES       = "/content/drive/MyDrive/Data Mining - Final Project/recipes_spices.csv"


# ---------- LOAD & PREPARE SPICE MAP ----------
# Load spice mapping
with open(SPICE_MAP_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)  # keys = canonical names, values = list of synonyms

# Helper to generate word forms
def _forms(s: str):
    """Generate common word forms for a spice (plural, hyphen/space variants, ies/ves endings, etc)."""
    s = s.strip().lower()
    out = {s}
    if " " in s: out.add(s.replace(" ", "-"))
    if "-" in s: out.add(s.replace("-", " "))
    if s.endswith("y") and not s.endswith(("ay","ey","iy","oy","uy")):
        out.add(s[:-1]+"ies")
    if s.endswith("f"): out.add(s[:-1]+"ves")
    if s.endswith("fe"): out.add(s[:-2]+"ves")
    if s.endswith("o") and not s.endswith(("oo","eo")):
        out.add(s+"es")
    if not s.endswith(("s","es","ies","ves")):
        out.add(s+"s")
    return out


# Expand spice synonyms with all possible forms
SPICE_MAP = {}
for canon, syns in raw.items():
    syns = syns or []
    base_syns = [canon, *syns]
    expanded = set(itertools.chain.from_iterable(_forms(x) for x in base_syns))
    SPICE_MAP[canon] = expanded

# Reverse lookup: synonym -> canonical name
SYN2CANON = {syn: canon for canon, syns in SPICE_MAP.items() for syn in syns}

# Regex Preparation
def _pat(term: str):
    """Create a regex pattern for a given spice synonym, allowing spaces, hyphens, or slashes as separators."""
    parts = [p for p in re.split(r"[\s\-/]+", term) if p]
    parts_esc = [re.escape(p) for p in parts]
    sep = r"(?:\s|[-/])+"
    return r"(?<!\w)" + sep.join(parts_esc) + r"(?!\w)"

TERMS = sorted(
    set(SYN2CANON.keys()) | {"pepper", "peppers"},  # include generic pepper only in regex
    key=len, reverse=True
)
REGEX = re.compile("|".join(_pat(t) for t in TERMS), re.IGNORECASE)

Now, we define helper functions to handle special cases that require attention in ingredient text, such as distinguishing garlic cloves from clove spice and resolving adjectives before 'pepper' (e.g., 'black pepper', 'red pepper') to the correct canonical spice name.

In [ ]:
# ---------- CONTEXT HELPERS ----------
# Special cases for handling garlic cloves and pepper
# Garlic cloves regex
CLV_G_AFTER = re.compile(r'(\bcloves?\b)(?:\s+\w+){0,3}\s+garlic\b', re.I)
CLV_G_BEFORE = re.compile(r'\bgarlic\b(?:\s+\w+){0,3}\s+(cloves?\b)', re.I)

def _ranges_overlap(a, b):
  """Check if two index ranges overlap."""
  return max(a[0], b[0]) < min(a[1], b[1])

# Handle "pepper"
def _find_adjacent_pepper(low_text: str, match_start: int):
    """Resolve bare 'pepper' by looking for an adjective immediately before it
    (e.g., 'black pepper'). If none exists, fallback to black pepper."""
    tokens = re.findall(r"\w+", low_text[:match_start])

    skip_mods = {"ground","powder","finely","coarsely","fresh","dried",
                 "crushed","whole","to","of","a","the","and","or"}

    meaningful = [t for t in tokens[-6:] if t not in skip_mods] if tokens else []

    # Try 2-word then 1-word adjectives right before 'pepper'
    for n in (2, 1):
        if len(meaningful) >= n:
            cand_adj = " ".join(meaningful[-n:])
            candidate = f"{cand_adj} pepper"
            if candidate in SYN2CANON:
                return SYN2CANON[candidate]

    # Fallback to black pepper only when no meaningful adjective precedes
    if not meaningful:
        return SYN2CANON.get("black pepper")

    return None

# Handle "clove"
def _accept_clove_match(matched_syn: str, low_text: str, span):
    """Return False if a 'clove' match actually refers to garlic cloves, True otherwise."""
    syn = matched_syn.lower().strip()
    if syn not in {"clove","cloves"}: return True
    a = CLV_G_AFTER.search(low_text)
    if a and _ranges_overlap(span, a.span(1)): return False
    b = CLV_G_BEFORE.search(low_text)
    if b and _ranges_overlap(span, b.span(1)): return False
    return True

Here we defines the main functions to extract spices from ingredient text and save each recipe’s spices to a CSV file.

In [ ]:
# ---------- SPICE MATCHING ----------
# Find spices in an ingredient string
def find_spices(ingredient: str):
    """Return a list of canonical spice names found in the ingredient string."""
    matches = set() # set first to avoid duplicates
    low = ingredient.lower()

    for m in REGEX.finditer(low):
        matched_syn = m.group(0).lower().strip()
        start, end = m.span()

        # Handle generic 'pepper' first so we don't get skipped by the canon check
        if matched_syn in {"pepper", "peppers"}:
            resolved = _find_adjacent_pepper(low, start)
            if resolved:
                matches.add(resolved)
            continue

        # Normal path
        canon = SYN2CANON.get(matched_syn)
        if not canon:
            continue

        # Filter false garlic clove matches
        if not _accept_clove_match(matched_syn, low, (start, end)):
            continue

        matches.add(canon)

    return list(matches)


# ---------- MAIN ----------
tqdm.pandas()  # enable progress_apply

def build_spice_csv():
    """Load ingredient CSV, extract spices per recipe, and save results to a CSV: id | cuisine | recipe_name | recipe_url | ingredient | spice"""

    df = pd.read_csv(RECIPE_INGS_PATH)

    # Group ingredients by recipe
    grouped_df = df.groupby(['id','cuisine','recipe_name','recipe_url'])['ingredient'].apply(list).reset_index()

    rows = []

    # Extract spices per recipe with progress bar
    for _, row in tqdm(grouped_df.iterrows(), total=len(grouped_df), desc="Extracting spices"):
        ingredients = row['ingredient']
        spices_found = set()
        spice_to_ingredient = {}

        # Go through ingredients to find spices
        for ing in ingredients:
            for spice in find_spices(str(ing)):
                if spice not in spices_found:
                    spices_found.add(spice)
                    spice_to_ingredient[spice] = ing  # remember ingredient where spice was found

        # Only keep recipes with at least one spice
        if spices_found:
            for spice, ing in spice_to_ingredient.items():
                rows.append({
                    'id': row['id'],
                    'cuisine': row['cuisine'],
                    'recipe_name': row['recipe_name'],
                    'recipe_url': row['recipe_url'],
                    'ingredient': ing,
                    'spice': spice
                })

    # Save to CSV
    df_out = pd.DataFrame(rows)
    df_out.to_csv(OUT_SPICES, index=False, encoding="utf-8")

    print(f"\nDone. Recipes processed: {len(set([r['id'] for r in rows]))}")
    print(f"Saved: {OUT_SPICES}")


# ---------- RUN ----------
if __name__ == "__main__":
    build_spice_csv()

Extracting spices: 100%|██████████| 5000/5000 [01:07<00:00, 74.07it/s]



Done. Recipes processed: 3902
Saved: /content/drive/MyDrive/Data Mining - Final Project/recipes_spices.csv


In [ ]:
df = pd.read_csv(OUT_SPICES)

# count values
recipes_num = df["id"].nunique()
unique_cuisines = df["cuisine"].nunique()
unique_spices = df["spice"].nunique()

print(f"Number of recipes with spices: {recipes_num}")
print(f"Unique cuisines: {unique_cuisines}")
print(f"Unique spices: {unique_spices}")

Number of recipes with spices: 3902
Unique cuisines: 100
Unique spices: 72


We received 3902 out of 5000 recipes that were identified with spices, with a total of 72 different spices and 100 different cuisines.